# LA 03d — Read-only VD18 volatility engine into Liquidity Absorption risk

**Research test, not an investable strategy.** A02 provides signal direction and level; A03/A03c provide first-post-close aggressive or hypothetical passive third-level-revisit entry. **VD18 determines a protective volatility floor on the stop, nominal position budget and horizon diagnostics.** All structural profit targets are held fixed by contract. Run from repo using `deribit-data` kernel.

This uses TRAIN-frozen `VD18 HAR_4` at 15, 60, 240 minutes. It is a forecast of **future terminal SPOT return variance**, not the maximum adverse perp price excursion. The risk transform is exploratory, including transfer from spot to perpetuals. It may still show negative net expectancy. The already-inspected Jul–Sep 2026 period cannot establish new predictive confirmation.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image
REPO = Path.cwd().resolve()
while not (REPO / "src" / "data_engineering").is_dir():
    if REPO == REPO.parent:
        raise RuntimeError("Launch notebook from repository tree")
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0,str(REPO))
from src.data_engineering.modeling.liquidity_absorption_stage_a import binance_root
from src.data_engineering.modeling.liquidity_absorption_stage_a03d import (
    VERSION, RiskConfig, build_results, spot_history_directory)
assert VERSION == "LA-A03d-0.1.0", "git pull; restart kernel"
DATA_ROOT = binance_root(REPO)
SPOT = spot_history_directory(REPO)
OUT = REPO / "notebooks" / "Liquidity_Absorption" / "03d"
CFG = RiskConfig()
print("Risk bridge:",VERSION,"| Futures:",DATA_ROOT,"| Spot:",SPOT)
print("NO trade directions or structural targets are predicted by VD18")

In [ ]:
results = build_results(REPO, DATA_ROOT, SPOT, OUT, CFG)
print("A03d result:", results)
assert results["n_events"]==452


## Model timestamps / causal data coverage

Every signal must use only the **exact most recent completed five-minute spot bar**. No forecast is filled from future bars or from a stale spot return window. Missing history is an explicit no-forecast status. We show the forecast sigma alongside the resulting stop-risk in perp basis points.


In [ ]:
events = pd.read_csv(OUT/"03d_event_forecasts.csv")
trades = pd.read_csv(OUT/"03d_risk_execution.csv")
summary = pd.read_csv(OUT/"03d_summary.csv")
paired = pd.read_csv(OUT/"03d_paired_summary.csv")
display(events.groupby(["symbol","forecast_status"]).size().rename("events").reset_index())
display(events.groupby("symbol")[["sigma_bp_15","sigma_bp_60","sigma_bp_240"]].describe().round(3))
print("Coverage, risk-aware costs/returns by policy and holding horizon")
display(summary)
print("240m, paired closed-trade baseline comparison in price basis points")
display(paired)
print("Recorded order outcomes and absence of confirmed fills")
display(trades.groupby(["entry_kind","model_mode","hold_minutes","reason"]).size().rename("n").reset_index())

## Virtual equity and stop-distance plots

The virtual P&L curve applies a constant **hypothetical risk budget** per trade, capped by reference notional. Trades can overlap; this is **not an account equity curve**. Maker-eligibility scenarios are not known maker fills and remain conditional on unavailable quote/queue state.


In [ ]:
for name in ("03d_virtual_equity.png","03d_forecast_vs_stop.png"):
    pic=OUT/name
    if pic.exists():
        print(name)
        display(Image(filename=str(pic)))

## Decision gate

1. This module deliberately does **not** optimize sigma multipliers or 60/240-minute holding limits using already-inspected July–September outcomes.
2. Forecast sigma is a terminal-variance measure; wick protection and liquidity invalidation must remain, since sigma alone does not imply stop-first probability.
3. The source spot price window is independent of perp execution, so spot/perpetual variance transfer remains an empirical question.
4. Stop sizing, fee allowance, virtual dollar risk and maker fill proxies are **not proof of live profitability**, portfolio capital adequacy, fill probability or actual funding-adjusted returns.
5. Earlier A03, A03b, A03c and Volatility Dynamics outputs are left untouched. Push executed notebook and `03d/` CSV/JSON/PNG artifacts for review.
